# 🔌 Utility Asset Detector

**One-click setup for detecting T&D utility infrastructure.**

This notebook runs on Google Colab with a free GPU. Detects:
- Utility poles (wood, concrete, steel)
- Transmission towers
- Insulators, transformers, cross-arms
- Damage: cracks, rust, woodpecker holes, lean, etc.

**Instructions:**
1. Click `Runtime → Run all` (or Ctrl+F9)
2. Wait ~3 minutes for installation
3. Use the Gradio interface that appears

---

In [ ]:
#@title 1️⃣ Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(f"\n✅ PyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title 2️⃣ Install Dependencies (~2 min)
%%capture
!pip install -q gradio pillow opencv-python pyyaml

# Clone and install DART
!git clone -q https://github.com/mkturkcan/DART.git
%cd DART
!pip install -q -e .
%cd ..

# Clone utility-asset-detector
!git clone -q https://github.com/menonpg/utility-asset-detector.git
%cd utility-asset-detector

print("✅ Installation complete!")

In [ ]:
#@title 3️⃣ Download SAM3 Checkpoint (~1.7 GB, first run only)
# The checkpoint auto-downloads on first inference
# This cell pre-downloads it to avoid timeout during detection

from huggingface_hub import hf_hub_download
import os

if not os.path.exists("sam3.pt"):
    print("Downloading SAM3 checkpoint...")
    hf_hub_download(
        repo_id="facebook/sam3-hiera-large",
        filename="sam3.pt",
        local_dir="."
    )
    print("✅ Checkpoint downloaded!")
else:
    print("✅ Checkpoint already exists")

In [ ]:
#@title 4️⃣ Launch Web UI 🚀
import sys
sys.path.insert(0, ".")

from app import create_ui

app = create_ui()
app.launch(
    share=True,  # Creates public URL
    debug=True,
)

---

## 📝 Alternative: Run Detection Directly

If you prefer code over the UI:

In [ ]:
#@title Direct Detection (Alternative)
from src.detector import UtilityAssetDetector
from PIL import Image
import requests
from io import BytesIO

# Initialize detector
detector = UtilityAssetDetector(
    config="configs/assets.yaml",
    device="cuda"
)

# Load test image
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8a/Wooden_utility_pole.jpg/800px-Wooden_utility_pole.jpg"
response = requests.get(url)
image = Image.open(BytesIO(response.content))

# Run detection
result = detector.detect(image)

# Print results
print(f"Structures found: {result.total_structures}")
print(f"Components found: {result.total_components}")
print(f"Priority: {result.priority}")
print()

for struct in result.structures:
    print(f"📍 {struct.type} ({struct.confidence:.0%})")
    print(f"   Status: {struct.condition.status}")
    if struct.condition.issues:
        print(f"   Issues: {', '.join(struct.condition.issues)}")
    for comp in struct.components:
        print(f"   - {comp.type}: {comp.condition.status}")

In [ ]:
#@title Save JSON Results
# Save detection results to file
with open("detection_result.json", "w") as f:
    f.write(result.to_json())

print("✅ Saved to detection_result.json")
print(result.to_json())